In [6]:
from train import *

ModuleNotFoundError: No module named 'train'

In [2]:
if __name__ == '__main__':
    batch_size, num_workers, num_epochs, = 32, 0, 200
    lr_ANN, middle_dim, num_layers_ANN = 0.1, 30, 5
    lr_KAN, num_layers_KAN, middle_dim_kan, dropout_p, degrees, weight_decay = 0.2, 3, 144, 0, [3, 5, 5, 5, 5], 1e-5
    FFT_grid_size = 32
    output_size = 1
    s_model_name = 'Heston'
    # s_model_name = 'FVSJ'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('当前设备', device)

    # cheby_kan神经网络
    # ===================================================================================================================
    train_loader, test_loader, pinn_loader, input_size = load_data(s_model_name=s_model_name, batch_size=batch_size, num_workers=num_workers, device=device)
    model = Cheby_KAN(input_size, output_size, middle_dim_kan, degrees, num_layers_KAN, dropout_p)
    loss_kan, kan_name = train_test(model, train_loader, test_loader, num_epochs, lr_KAN, weight_decay, device=device, net_name='PCKAN', pinn_params=pinn_loader, save_dir=s_model_name)
    params_list_kan = [[lr_KAN, num_layers_KAN, num_epochs, middle_dim_kan, degrees, num_epochs, loss_kan, dropout_p, weight_decay, kan_name]]
    params_pd_kan = pd.DataFrame(params_list_kan, columns=['Learning_Rate', 'Num_Layers', "train_epochs", "middle_dim", 'Degrees', 'Num_epochs', 'Loss', "dropout_p", "weight_decay", 'Net_Name'])
    params_pd_kan.to_csv('../data/train/train_params_kan_pinn_lll.csv', mode='a', index=False)
    # ===================================================================================================================

    # 全连接
    # ==================================================================================================================
    train_loader, test_loader, pinn_loader, input_size = load_data(s_model_name=s_model_name, batch_size=batch_size, num_workers=num_workers, device=device)
    model = nn_impvol(input_size, output_size, middle_dim, num_layers_ANN)
    loss_ann, ann_name = train_test(model, train_loader, test_loader, num_epochs, lr_KAN, weight_decay, device=device, net_name='ANN', pinn_params=pinn_loader, save_dir=s_model_name)
    params_list_kan = [[lr_ANN, num_layers_ANN, num_epochs, middle_dim, num_epochs, loss_ann, dropout_p, weight_decay, ann_name]]
    params_pd_kan = pd.DataFrame(params_list_kan, columns=['Learning_Rate', 'Num_Layers', "train_epochs", "middle_dim", 'Num_epochs', 'Loss', "dropout_p", "weight_decay", 'Net_Name'])
    params_pd_kan.to_csv('../data/train/train_params_ann_lll.csv', mode='a', index=False)
    # ===================================================================================================================

    # kan神经网络
    # ===================================================================================================================
    train_input, train_label, test_input, test_label = load_data(type="normal_kan")
    model = KAN(width=[9, 5, 5, 5, 9], grid=3, k=3, seed=42, device=device)
    dataset = {
        "train_input": train_input,
        "train_label": train_label,
        "test_input": test_input,
        "test_label": test_label}
    model(dataset['train_input'])
    model.fit(dataset, opt="LBFGS", steps=200, lamb=0.001)
    # ===================================================================================================================

当前设备 cuda


NameError: name 'load_data' is not defined

In [1]:
import torch
torch.cuda.is_available()

True